# NB02: Data Transformation

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250093214


## Setup

Run the cell below to ensure all required packages are installed before running all other cells.

In [1]:
import json
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


## Storing the collected data in dataframes

Now that the data has been collected and safely stored, it is time to arrange the data into neat dataframes.

### How many dataframes will be created?
There are going to be two different dataframes: one containing the species data of the Pokemon, and one containing the actual stats data of the Pokemon.

### What will each dataframe contain?
For the dataframe with the species data, each row is going to contain the ID of the Pokemon, the name of the Pokemon, the flags of whether the Pokemon is a baby, a legendary, or a mythical, and the Generation that the Pokemon is associated with. I will also include a BST for each Pokemon created from the data contained in the stats dataframe. 

For the dataframe with the stats data, each row is going to contain the ID, name of the Pokemon, and its associated Generation once again, and all of their individual base stats and the names of those stats.

### Why two different dataframes?

There are a couple of reasons why two different dataframes are needed:
- Having the BST and the individual stats data separated out will make for easier analysis later, as I can choose to combine the dataframes with pd.merge() if the need arises, or I can keep them separate and perform individual analyses on the data they contain
- The dataframe containing the stats data has six rows for each Pokemon (one for each kind of base stat), while the dataframe containing the species data has one row for each Pokemon, so attempting to combine them out of the gate will lead to misaligned dataframes and bad data
- The data is stored in two different API calls, so creating two different dataframes is easier for me, personally

In [3]:
print(api_calls)

NameError: name 'api_calls' is not defined

In [ ]:
spec_df = ''
with open(f'../data/raw/{mon_name[0]}_spec.json', mode='r') as f:
        data = json.load(f)
        df = pd.json_normalize(
            data,
            record_path = 'varieties',
            meta = ['id', 'name', 'is_baby', 'is_legendary', 'is_mythical', 'forms_switchable','has_gender_differences'],
            sep='_'
        )
        gen_df = pd.DataFrame(data['generation'],index=[0]).rename(columns={'name':'gen', 'url':'gen_url'})

initial_df = pd.concat([df,gen_df],axis=1)


for i in range(1, len(api_calls)):
        with open(f'../data/raw/{mon_name[i]}_spec.json', mode='r') as f:
                data = json.load(f)
                df = pd.json_normalize(
                data,
                record_path = 'varieties',
                meta = ['id', 'name', 'is_baby', 'is_legendary', 'is_mythical', 'forms_switchable','has_gender_differences'],
                sep='_'
                )
                gen_df = pd.DataFrame(data['generation'],index=[0]).rename(columns={'name':'gen', 'url':'gen_url'})
        dfc = pd.concat([df,gen_df],axis=1)
        main_df = pd.concat([initial_df,dfc],axis=0, ignore_index = True)
        initial_df = main_df

In [ ]:
for i in range(len(api_calls)):
        with open(f'../data/raw/{mon_name[i]}_spec.json', mode='r') as f:
                data = json.load(f)
                df = pd.json_normalize(
                data,
                record_path = 'varieties',
                meta = ['id', 'name', 'is_baby', 'is_legendary', 'is_mythical', 'forms_switchable','has_gender_differences'],
                sep='_'
                )
                gen_df = pd.DataFrame(data['generation'],index=[0]).rename(columns={'name':'gen', 'url':'gen_url'})
        dfc = pd.concat([df,gen_df],axis=1)
        main_df = pd.concat([initial_df,dfc],axis=0, ignore_index = True)
        initial_df = main_df

.rename(): I realized that when I was creating the separate data frame for the Pokemon's generation data, some of the column names were being repeated (ex: 'name' being used for both the name of the Pokemon species and the name of the generation where the Pokemon originated from). As such, I decided to use the .rename function, which takes a dictionary using the original names as keys and the new names as the values. You can rename either columns or indexes with these, but I chose to rename columns to eliminate duplicate names.

I need to create two separate data frames for my ideal analysis: one with the BSTs, and one with the separate base stats. For the BSTs, I am going to create a data frame with just the BSTs and concatenate that on my other data frame. For the separate base stats, however, I need to create a data frame and add the Generation of the Pokemon on manually, due to each Pokemon having six rows (I cannot easily concatenate this information onto my other data frame with this row disparity.) As such, I will create a function that will assign the correct Generation to each Pokemon by matching the Pokemon's name to the Generation that they came from, then apply this function to every row on the separate base stats data frame.

In [ ]:
## Total BST
with open(f'../data/raw/{form_name[0]}_stat.json', mode='r') as f:
    data = json.load(f)
    dfi = pd.json_normalize(
        data,
        record_path = 'stats',
        sep='_'
    )
    dfi = dfi.assign(BST = dfi['base_stat'].sum()).drop(index = range(1,6)).drop(columns = ['base_stat','effort','stat_name','stat_url'])

for i in range(1, len(stat_calls)):
    with open(f'../data/raw/{form_name[i]}_stat.json', mode='r') as f:
            data = json.load(f)
            df = pd.json_normalize(
                data,
                record_path = 'stats',
                sep='_'
            )
            df = df.assign(BST = df['base_stat'].sum()).drop(index = range(1,6)).drop(columns = ['base_stat','effort','stat_name','stat_url'])
    BST_df = pd.concat([dfi,df],axis=0, ignore_index = True)
    dfi = BST_df

## Separate Base Stats
with open(f'../data/raw/{form_name[0]}_stat.json', mode='r') as f:
    data = json.load(f)
    dfi = pd.json_normalize(
        data,
        record_path = 'stats',
        meta = ['id', 'name'],
        sep='_'
    )

for i in range(1, len(stat_calls)):
    with open(f'../data/raw/{form_name[i]}_stat.json', mode='r') as f:
            data = json.load(f)
            df = pd.json_normalize(
                data,
                record_path = 'stats',
                meta = ['id','name'],
                sep='_'
            )
    stats_df = pd.concat([dfi,df],axis=0, ignore_index = True)
    dfi = stats_df

stats_df = stats_df.drop(columns = ['effort', 'stat_url'])

In [ ]:
def discover_generation(pokemon_name):
    for gen in gen_list:
        if pokemon_name in gen_mon[gen]:
            return gen

In [ ]:
main_df = pd.concat([main_df,BST_df],axis=1)

main_df

In [ ]:
main_df.to_csv('../data/processed/main_df.csv', index = False)

stats_df.to_csv('../data/processed/stats_df.csv', index = False)a

## What Pokemon will be included in the analysis?

Pokemon has introduced a lot of gimmicks in its 30 years of existence, and many of these gimmicks affect Pokemon: Mega Evolution, Dynamax and Gigantamax, items that transform a Pokemon, and many others. These gimmicks often directly affect a Pokemon's stats. The question is, how do we factor in these gimmicks when deciding what Pokemon to use for analysis?

# Rewrite this according to what I actually filtered out!!!

All standard Pokemon will be included in the analysis. Pokemon who have different forms (ex: Wormadam, who has three different stat spreads depending on whether it is a Plant, Sandy, or Trash cloak) or experience gender differences (ex: Nidoran, Basculegion, Indeedee) that **cannot be switched between** will be counted as a separate Pokemon species. Pokemon that experience purely visual differences are already filtered out by the API, so there is no need to consider them when transforming the data. Pokemon that have forms that can be swtiched between that don't have different BSTs (ex: Aegislash has a Shield Forme and a Blade Forme that it can switch to basically at will during a battle that affects its stat spread but not its BST) will only have one instance counted. Pokemon that have different forms that can be switched between that **do** affect BSTs but **are not due to a gimmick** (Mega Evolution, Dynamax, Terastallization) will only have their highest BST form counted, as this is likely the form that players are both actually using the most and how GameFreak (the developers of Pokemon) intended for this Pokemon to be used (ex: Zacian and Zamazenta will have only their Crowned forms counted; Calyrex will have both of its Ice Rider and Shadow Rider forms counted but not Calyrex, Glastrier, and Spectrier alone; Terapagos will have its Terastal Form counted rather than its Stellar Form because its Stellar Form can only be accessed through a gimmick, etc. etc). This distinction also excludes Mega and Gigantamax Pokemon. 

I would also like to examine BSTs by categories of Pokemon, which in this case are Standard, Legendary, and Mythical Pokemon. These categories have been chosen because they are categories that the games themselves use to categorize Pokemon. Additionally, all three categories recieve new Pokemon with each new Generation. These categories will follow the same filter guidelines as above, but there will be additional filters imposed onto it: for Legendaries, only the fully-evolved version of the Legendary will be included (ex: No Cosmog or Cosmoem, only Zygarde's Complete Forme), as Legendary Pokemon don't typically evolve and when they do evolve, they are intended to be used only in their fully-evolved state. It makes the comparison between all Legendaries for all Generations fairer in this way. The same construct also applies for Mythicals. The fully-evolved rule does not count for Standard Pokemon, since evolution is very common for Pokemon in this class. Pokemon considered "Baby Pokemon" will be filtered out, as this is its own class of Pokemon that does not receieve new additions every Generation, so there is not enough data over time to compare to. Pokemon that are technically a special class of Pokemon introduced for one Generation (ex: the Ultra Beasts from Generation 7, the Paradox Pokemon from Generation 9) will be included under the Standard Pokemon category. Initially, I was going to filter them out, but upon further consideration, introducing a new more-powerful subclass of Standard Pokemon for one Generation that does not become its own unique class is a good indicator of power creep, as GameFreak felt the need to make more 'special' Standard Pokemon.

Pokemon not included in base generation games but introduced later (ex: in deluxe re-releases, in DLC) will be included, subject to the filter guidelines explained above.

## Filtering out Pokemon according to the guidelines above

In order to filter out the Pokemon that I do not want to include (Megas, Gigantimaxes, etc.), I am going to use a pandas function called .notna(). This will filter out all of the rows that do not contain data in one of the columns; in this case, the 'Generation' (Gen) column. If a Pokemon does not match their base species (such as, for example, being a Mega), there will not be Generation data attached to them. Filtering out all of the Pokemon with no Generation data will filter out all of the undesired Pokemon from my analysis, though it will have the unfortunate side-effect of weakening some of the later Generation Legendaries due to their weaker forms being considered the 'base' species (as in, that is how the Pokemon exists by default). This is a trade-off I am willing to make, however, when it comes to comparing Generations as a whole, because it removes many Pokemon that would skew the data. For instance, Pikachu has many costumes that it can have in the games that give it no BST or base stat differences, so including all of these Pokemon in the Generation 1 category would heavily skew the data by treating costumes as new species. Removing the higher-BST versions of various legendaries is a worthy trade-off.

This trade-off is likely to cause problems, however, when looking at how BSTs and base stats overall have changed within the three different categories due to there being less data in the Legendaries and Mythicals categories to balance out using the lesser-version of various Legendaries.

Once I've done the filter on the dataset containing the BSTs, I am going to perform a .merge() on my two datasets to join them together. I am using a .merge() because my dataset containing the BSTs is not the same length as the dataset containing each individual stat, but they have similar keys (specifically in the 'name' column), so I can join them together. I am doing a left join, joining my BST dataset with my individual dataset, which I am doing to make sure I capture all 1025 Pokemon left after the filter was applied. 

In [13]:
# start = pd.read_csv('../data/processed/main_df.csv')

# def string_split(df):
#     for value in df['name']:

#         multi_info = df['name'].str.split('-')

#         df['form_name'] = multi_info[0]
#         if len(multi_info) >= 2
#         df['form'] = df['name'].str.split('-')[1]
#         df['extra1'] = df['name'].str.split('-')[2]
#         df['extra2'] = df['name'].str.split('-')[3]
#     return dup_forms

# start_split = string_split(start)

In [14]:
start = pd.read_csv('../data/processed/main_df.csv')

base_table = start[start['gen'].notna()]

stats_df = pd.read_csv('../data/processed/stats_df.csv')

stats_table = pd.merge(
    base_pkmn,
    stats_df,
    how = 'left',
    on = 'name'
)


Now, I have to transform the generations into numbers instead of roman numerals, because this method of ordering generations will cause issues when it comes to plotting my chart.

In [15]:
gen_dict = {
    'generation-i':'Gen 1',
    'generation-ii':'Gen 2',
    'generation-iii':'Gen 3',
    'generation-iv':'Gen 4',
    'generation-v':'Gen 5',
    'generation-vi':'Gen 6',
    'generation-vii':'Gen 7',
    'generation-viii':'Gen 8',
    'generation-ix':'Gen 9'
}

In [16]:
def number_gen(generation):
    for key in gen_dict.keys():
        if generation in key:
            return gen_dict[key]

In [17]:
number_gen('generation-i')

'Gen 1'

In [18]:
stats_table['gen_num'] = stats_table['gen_x'].apply(number_gen)
base_table['gen_num'] = base_table['gen'].apply(number_gen)

Now that my final data table has been created, I can store it in a .csv file for usage in my data analysis.

In [19]:
stats_table.to_csv('../data/processed/stats_table.csv', index=False)
base_table.to_csv('../data/processed/base_table.csv', index=False)
